# पाठ ११ - एजेन्ट-देखि-एजेन्ट (A2A) प्रोटोकल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A प्रोटोकल भनेको के हो?

यो **Agent-to-Agent (A2A) प्रोटोकल** एउटा खुला मानक हो जसले एआई एजेन्टहरूलाई संवाद गर्न, एक अर्कालाई पत्ता लगाउन, र सहयोग गर्न सक्षम बनाउँछ — यहाँसम्म कि जब तिनीहरू फरक फ्रेमवर्कमा बनेका वा फरक सेवाहरूमा होस्ट गरिएको हुन्छन्।

Key concepts:

- **Discovery** – एजेन्टहरूले आफ्नो क्षमताहरू वर्णन गर्ने *एजेन्ट कार्ड* प्रकाशन गर्छन्, जसले अन्य एजेन्टहरू (वा ओर्केस्ट्रेटरहरू) लाई कुनै कार्यका लागि सही विशेषज्ञ फेला पार्न सजिलो बनाउँछ।
- **Message Passing** – एजेन्टहरूले साझा प्रोटोकलबाट संरचित सन्देशहरू आदानप्रदान गर्छन्, त्यसैले एक एजेन्टको अनुरोध अर्कोले यसको आन्तरिक कार्यान्वयन कस्तो छ भन्ने कुराबाट स्वतन्त्र भएर बुझ्न र पूरा गर्न सक्छ।
- **Task Lifecycle** – A2A ले *submitted*, *working*, *completed*, र *failed* जस्ता अवस्थाहरू परिभाषित गर्छ, जसले ओर्केस्ट्रेटरलाई हस्तान्तरण गरिएको कार्य कसरी प्रगति गर्दैछ भन्ने पूर्ण दृश्यता दिन्छ।

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## विशेषीकृत यात्रा एजेन्टहरू सिर्जना गर्दै


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ्लो मार्फत बहु-एजेन्ट सहकार्य

हामी ती तीन एजेन्टहरूलाई A2A सन्देश आदानप्रदानलाई प्रतिबिम्बित गर्ने एउटा अनुक्रमिक वर्कफ्लोमा जडान गर्छौं:

1. **CurrencyExchangeAgent** प्रयोगकर्ताको अनुरोध प्राप्त गर्छ र मुद्रा सम्बन्धी मार्गदर्शन उत्पादन गर्छ।
2. **ActivityPlannerAgent** समृद्ध सन्दर्भ प्राप्त गर्छ र गतिविधि सिफारिशहरू थप्छ।
3. **TravelManagerAgent** दुवै इनपुटहरूलाई अन्तिम यात्रा संक्षेपमा समेट्छ।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## उत्पादन वातावरणमा A2A बुझाइ

उत्पादन वातावरणमा A2A प्रोटोकलले शक्तिशाली क्रस-सर्भिस परिदृश्यहरू खोल्छ:

| Capability | Description |
|---|---|
| **क्रस-फ्रेमवर्क अन्तरक्रिया** | एउटै फ्रेमवर्कमा बनेको एजेण्टले कार्यहरू अरू कुनै पनि A2A-अनुकूल फ्रेमवर्कमा बनेको एजेण्टलाई सुम्पन सक्छ, जसले साँच्चिकै संस्था-बीच अन्तरक्रियाशीलता सक्षम बनाउँछ। |
| **सेवा सीमाहरू** | एजेण्टहरू अलग-अलग माइक्रोसर्भिसहरू, क्लाउड क्षेत्रहरू, वा भिन्न संस्थाहरूमा बसोबास गर्न सक्छन् र तैपनि सहजतापूर्वक सहकार्य गर्न सक्छन्। |
| **गतिशील खोज** | एक ऑर्केस्ट्रेटरले रनटाइममा Agent Card रजिस्ट्रीलाई सोध्दै दिइएको उप-कार्यका लागि सबैभन्दा उपयुक्त विशेषज्ञ पत्ता लगाउन सक्छ। |
| **स्ट्रीमिङ र पुश सूचनाहरू** | A2A ले Server-Sent Events (SSE) मार्फत वास्तविक-समय प्रगति अपडेटहरू र लामो समयसम्म चल्ने कार्यहरूको लागि पुश सूचनाहरू समर्थन गर्छ। |

हामीले माथि बनाएको वर्कफ्लो यो ढाँचाको एक सरल, इन-प्रोसेस संस्करण हो। वास्तविक
परिनियोजनमा प्रत्येक एजेण्टले एक HTTP endpoint खुलाउने, Agent Card प्रकाशित गर्ने, र सञ्चार गर्ने
A2A JSON-RPC प्रोटोकल मार्फत।


## सारांश

यस पाठमा तपाईंले सिक्नुभयो:

1. **A2A प्रोटोकल के हो** — एजेन्ट-देखि-एजेन्ट खोज, सन्देश,
   र कार्य व्यवस्थापनका लागि एक खुला मानक।
2. **विशेषीकृत एजेन्टहरू कसरी सिर्जना गर्ने** — एक Currency Exchange agent, an Activity Planner agent, र एक Travel Manager orchestrator।
3. **एजेन्टहरूलाई कार्यप्रवाहमा कसरी जोड्ने** — एजेन्टहरूबीच क्रमिक सन्देश पारगमनको मोडेल बनाउन `WorkflowBuilder` प्रयोग गर्दै।
4. **A2A उत्पादनमा कसरी काम गर्छ** — गतिशील खोज र स्ट्रिमिङ अद्यावधिकहरूसँग फ्रेमवर्कहरू तथा सेवाहरूबीचको सहयोग सक्षम बनाउँदै।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यस कागजातलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धतामा प्रयास गर्छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा असंगतता हुन सक्छ। मूल भाषामा रहेको दस्तावेजलाई अधिकारिक स्रोत मानिनु पर्छ। महत्वपूर्ण जानकारीको लागि पेशेवर मानवीय अनुवाद सिफारिस गरिन्छ। यो अनुवादको प्रयोगबाट हुने कुनै पनि भ्रम वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
